# Advanced Session: Cypher Graph Practice
AI for Materials Science — Hands-on session 3

Query the shared battery graph and explore the matching nodes and relationships.

**Result:** Your Cypher query and a graph screenshot.

Run top to bottom. No API key needed.

[Cypher documentation](https://neo4j.com/docs/cypher-manual/5/introduction/)

`MATCH`: pattern · `WHERE`: condition · `RETURN p`: graph path · `LIMIT`: result count.
Write Cypher below `%%cypher`; run to draw. Zoom in for labels; click nodes or relationships for properties.

In [ ]:
#@title 1. Setup — run once; expand to inspect helper code
%pip -q install neo4j neo4j-viz==1.6.0 pandas numpy gdown


import json

import re

import subprocess

import time


import urllib.request


from pathlib import Path


import pandas as pd


def download_file(url, path):
    import gdown
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if not path.exists():
        file_id = url.split('/d/')[1].split('/')[0]
        gdown.download(id=file_id, output=str(path), quiet=True)
    if not path.exists() or path.read_bytes().lstrip().startswith(b'<'):
        raise ValueError('Download failed. Upload the supplied data file from the course folder instead.')
    return path

def class_graph(folder):
    url = 'https://drive.google.com/file/d/1aEQu4OSu6TqrXWpcVaLNYAvXmJ-PbD9s/view?usp=sharing'
    path = download_file(url, Path(folder) / 'class_graph.json')
    return json.loads(path.read_text(encoding='utf-8'))

def start_neo4j():
    """A dedicated local teaching database; no Neo4j account or external server."""
    import tarfile
    from neo4j import GraphDatabase
    home = Path('/opt/neo4j')
    if not (home / 'bin/neo4j').exists():
        subprocess.run(['apt-get', '-qq', 'install', '-y', 'openjdk-21-jre-headless'], check=True, stdout=subprocess.DEVNULL)
        archive = '/tmp/neo4j-lab.tgz'
        urllib.request.urlretrieve('https://dist.neo4j.org/neo4j-community-5.26.20-unix.tar.gz', archive)
        with tarfile.open(archive) as tar:
            tar.extractall('/opt', filter='data')
        Path('/opt/neo4j-community-5.26.20').rename(home)
        with (home / 'conf/neo4j.conf').open('a') as f:
            f.write('\ndbms.security.auth_enabled=false\nserver.default_listen_address=127.0.0.1\nserver.memory.heap.max_size=1G\nserver.memory.pagecache.size=256m\n')
    subprocess.run([str(home / 'bin/neo4j'), 'start'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    driver = GraphDatabase.driver('bolt://127.0.0.1:7687', auth=None)
    for _ in range(60):
        try:
            driver.verify_connectivity()
            return driver
        except Exception:
            time.sleep(2)
    raise RuntimeError('Neo4j did not start. Restart the runtime and run the setup again.')

def load_graph(graph, driver):
    with driver.session() as session:
        session.run('MATCH (n:LabEntity) DETACH DELETE n').consume()
        session.run('CREATE CONSTRAINT IF NOT EXISTS FOR (n:LabEntity) REQUIRE n.id IS UNIQUE').consume()
        for kind in sorted({n['type'] for n in graph['nodes']}):
            if not re.fullmatch(r'[A-Z][A-Za-z0-9]*', kind):
                raise ValueError(f'Invalid node type: {kind}')
            rows = [n for n in graph['nodes'] if n['type'] == kind]
            session.run(f'UNWIND $rows AS row CREATE (n:LabEntity:`{kind}`) SET n.id="ms49900:"+row.id, n.name=row.label, n.type=row.type, n.papers=row.papers, n.n_papers=row.n_papers', rows=rows).consume()
        for kind in sorted({e['relation'] for e in graph['edges']}):
            if not re.fullmatch(r'[A-Z][A-Z0-9_]*', kind):
                raise ValueError(f'Invalid relation type: {kind}')
            rows = [e for e in graph['edges'] if e['relation'] == kind]
            session.run(f'UNWIND $rows AS row MATCH (a:LabEntity {{id:"ms49900:"+row.source}}), (b:LabEntity {{id:"ms49900:"+row.target}}) CREATE (a)-[r:`{kind}`]->(b) SET r.evidence=row.evidence, r.paper=row.paper_id, r.page=row.page, r.condition=row.condition', rows=rows).consume()
    print(f"Neo4j ready: {len(graph['nodes'])} nodes / {len(graph['edges'])} relationships")

def show_cypher(driver, cypher, **params):
    if not any(line.strip() and not line.lstrip().startswith('//') for line in cypher.splitlines()):
        print('Enter your Cypher below %%cypher, then run this cell.')
        return
    from neo4j import Result
    from neo4j_viz.neo4j import from_neo4j
    result = driver.execute_query(cypher, parameters_=params, result_transformer_=Result.graph)
    if not result.nodes:
        print('No matching graph. Use RETURN n or RETURN p to draw nodes or paths; check your filters.')
        return
    view = from_neo4j(result)
    for n in view.nodes:
        n.caption = str(n.properties.get('name', ''))[:45]
    return view.render(height='550px')

def enable_cypher():
    from IPython import get_ipython
    shell = get_ipython()
    def cypher(line, cell):
        return show_cypher(shell.user_ns['driver'], cell)
    shell.register_magic_function(cypher, magic_kind='cell')

enable_cypher()

pd.set_option('display.max_colwidth', 160)
print('Setup ready.')

## 2. Load the class graph
Use the supplied 100-paper graph, including `Paper`, `Cell`, `Material`, and `Property` nodes.

In [ ]:
graph = class_graph('.')
driver = start_neo4j()
load_graph(graph, driver)

## 3. Example: material → property
Run the cell. Change `LIMIT 20` to compare graph sizes.

In [ ]:
%%cypher
MATCH p=(m:LabEntity:Material)-[r]->(v:Property)
RETURN p
LIMIT 20

## 4. Example: filter by a keyword
Change `'retention'` to `'conductivity'` and rerun.

In [ ]:
%%cypher
MATCH p=(m:LabEntity:Material)-[r]->(v:Property)
WHERE toLower(v.name) CONTAINS 'retention'
RETURN p
LIMIT 15

## 5. Example: paper → cell → property
Follow two steps; select a relationship to read its evidence and PDF page.

In [ ]:
%%cypher
MATCH p=(paper:LabEntity:Paper)-[:REPORTS]->(cell:Cell)-[:SHOWS]->(v:Property)
RETURN p
LIMIT 10

## 6. Write your own Cypher query
Write your query below `%%cypher` in the next code cell. Return nodes or paths to display a graph. Copy one example at a time, then change its labels, keywords, or result limit.

### Quick syntax guide
- `MATCH`: find a node or relationship pattern.
- `(m:Material)`: a node with the label `Material`, assigned to the variable `m`.
- `m.name`: the `name` property of that node.
- `-[r]->`: an outgoing relationship; `-[r]-` matches either direction.
- `WHERE`: filter matches by a condition.
- `RETURN m` or `RETURN p`: return a node or a matched path for visualization.
- `LIMIT 10`: return at most 10 matching rows.

### Example 1: Find material nodes
Change `Material` to `Paper`, `Cell`, or `Property` to explore another node label.

```cypher
MATCH (m:LabEntity:Material)
RETURN m
LIMIT 10
```

### Example 2: Filter properties by a keyword
`toLower()` makes the text lowercase; `CONTAINS` checks for a substring. Try changing `'retention'` to `'conductivity'`.

```cypher
MATCH (v:LabEntity:Property)
WHERE toLower(v.name) CONTAINS 'retention'
RETURN v
LIMIT 10
```

### Example 3: Explore a material's neighbors
The pattern below matches a relationship in either direction. `p` stores the complete path, including both nodes and their relationship.

```cypher
MATCH p=(m:LabEntity:Material)-[r]-(neighbor:LabEntity)
RETURN p
LIMIT 15
```

### Example 4: Follow a two-step path
Use relationship types such as `REPORTS` and `SHOWS` to follow a specific pattern from a paper to a cell and then to a property.

```cypher
MATCH p=(paper:LabEntity:Paper)-[:REPORTS]->(cell:Cell)-[:SHOWS]->(v:Property)
WHERE toLower(v.name) CONTAINS 'retention'
RETURN p
LIMIT 10
```

If a keyword returns no matches, try another keyword or remove the `WHERE` line. Select a relationship in the graph to inspect its evidence and source page.

In [ ]:
%%cypher
// Write your Cypher here